# Bonus Task — Transfer Learning for Bharatanatyam Mudra Recognition

## Objective

Use **transfer learning with MobileNetV2** and compare it with the custom CNN from the original mudra classification task.

The comparison uses:

- Test accuracy
- Model size
- Training time
- Inference speed

The same train/validation/test split is used for both models so the comparison is fair.

> **Note:** This notebook is a separate project. It does not depend on the previous project files.


## 1. Import Libraries

The code is intentionally kept simple. We use PyTorch for model building and training, torchvision for the pretrained MobileNetV2 model, and scikit-learn for evaluation.

In [ ]:
import os
import zipfile
import time
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load the Dataset

Upload `mudra.zip` when running in Google Colab, or place it in the same folder as this notebook.

The expected structure inside the zip is:

```text
Bharatanatyam_Mudras/
    Musti/
    Pataka/
    Sikhara/
    Simhamukha/
    Trisula/
```

In [ ]:
# Change this path if your zip file is stored somewhere else
ZIP_PATH = "mudra.zip"
EXTRACT_PATH = "mudra_data"

if not os.path.exists(ZIP_PATH):
    print("mudra.zip was not found.")
    print("In Google Colab, upload the file using the Files panel and run this cell again.")
else:
    if not os.path.exists(EXTRACT_PATH):
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
        print("Dataset extracted.")
    else:
        print("Dataset folder already exists.")

# Find the folder containing the class folders
possible_roots = [
    os.path.join(EXTRACT_PATH, "Bharatanatyam_Mudras"),
    EXTRACT_PATH
]

DATA_DIR = None

for folder in possible_roots:
    if os.path.exists(folder):
        class_folders = [
            name for name in os.listdir(folder)
            if os.path.isdir(os.path.join(folder, name))
        ]
        if len(class_folders) >= 5:
            DATA_DIR = folder
            break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find the Bharatanatyam_Mudras dataset folder.")

print("Dataset path:", DATA_DIR)


## 3. Basic Dataset Analysis

Before training, check the number of images in each class.

### Observation

A balanced dataset gives each class a similar contribution during training. If one class has many more examples, the model can become biased toward that class and overall accuracy can hide poor performance on smaller classes.

In [ ]:
class_counts = {}

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if os.path.isdir(class_path):
        image_count = 0
        for file_name in os.listdir(class_path):
            if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
                image_count += 1
        class_counts[class_name] = image_count

counts_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["Mudra", "Images"]
)

display(counts_df)

plt.figure(figsize=(8, 4))
plt.bar(counts_df["Mudra"], counts_df["Images"])
plt.xlabel("Mudra")
plt.ylabel("Number of images")
plt.title("Class Distribution")
plt.xticks(rotation=20)
plt.show()


### Analytical Question: Is the dataset balanced?

The dataset contains approximately the same number of images for each of the five mudras. Therefore, class imbalance is not a major problem for this experiment.

If imbalance were present, the model could learn the majority class more often. This could increase overall accuracy while reducing recall for minority classes. Possible solutions would include class weights, oversampling, or collecting more examples for the smaller classes.

## 4. Train / Validation / Test Split

A stratified split is used so that every mudra keeps a similar class proportion in all three datasets.

- 70% training
- 15% validation
- 15% testing

Both models use exactly the same image split.

In [ ]:
base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
num_classes = len(class_names)

all_indices = np.arange(len(base_dataset))
all_labels = np.array([label for _, label in base_dataset.samples])

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=all_labels
)

temp_labels = all_labels[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Classes:", class_names)
print("Total images:", len(all_indices))
print("Training images:", len(train_idx))
print("Validation images:", len(val_idx))
print("Test images:", len(test_idx))


## 5. Image Preprocessing

MobileNetV2 was trained on ImageNet, so the transfer-learning model uses the ImageNet mean and standard deviation.

The custom CNN also uses normalized images so that both models receive the same basic input scale.

Training images receive small augmentations. Validation and test images are only resized and normalized.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

train_dataset_full = datasets.ImageFolder(DATA_DIR, transform=train_transform)
test_dataset_full = datasets.ImageFolder(DATA_DIR, transform=test_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(test_dataset_full, val_idx)
test_dataset = Subset(test_dataset_full, test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Data loaders created.")


## 6. Custom CNN Baseline

This is the simple CNN baseline used for the comparison.

The network has three convolution blocks followed by adaptive average pooling and a small fully connected layer. It is intentionally not made very deep because this is a student-level comparison.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


def create_custom_cnn():
    return CustomCNN(num_classes).to(device)


## 7. Training Function

The same training function is used for both models.

The best validation model is saved in memory so that the final test evaluation is done using the epoch with the best validation accuracy.

In [ ]:
def train_model(model, train_loader, val_loader, epochs, learning_rate):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_accuracy = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []
    }

    start_time = time.perf_counter()

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total

        model.eval()

        val_loss_total = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_total += loss.item() * images.size(0)

                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total
        val_accuracy = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

    training_time = time.perf_counter() - start_time

    model.load_state_dict(best_weights)

    return model, history, training_time


## 8. Train the Custom CNN

For a normal run, five epochs are enough to demonstrate the comparison. If a GPU is available, the notebook can be increased to 8–10 epochs for a stronger final model.

In [ ]:
CUSTOM_EPOCHS = 5
CUSTOM_LR = 0.001

custom_model = create_custom_cnn()

custom_model, custom_history, custom_training_time = train_model(
    custom_model,
    train_loader,
    val_loader,
    epochs=CUSTOM_EPOCHS,
    learning_rate=CUSTOM_LR
)

print(f"Custom CNN training time: {custom_training_time:.2f} seconds")


In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_accuracy"], label="Training Accuracy")
    plt.plot(epochs, history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_loss"], label="Training Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()


plot_history(custom_history, "Custom CNN")


## 9. Transfer Learning with MobileNetV2

MobileNetV2 is loaded with ImageNet pretrained weights.

The convolutional backbone is frozen first. Only the final classifier is trained for this comparison. This is called **feature extraction**: the pretrained network supplies useful visual features and the final layer learns the five mudra classes.

> The first execution may download the pretrained ImageNet weights. A working internet connection is therefore needed the first time this cell is run.

In [ ]:
# Load MobileNetV2 with ImageNet pretrained weights
weights = models.MobileNet_V2_Weights.DEFAULT

transfer_model = models.mobilenet_v2(weights=weights)

# Freeze the pretrained feature extractor
for parameter in transfer_model.features.parameters():
    parameter.requires_grad = False

# Replace the original ImageNet classifier
input_features = transfer_model.classifier[1].in_features

transfer_model.classifier[1] = nn.Linear(input_features, num_classes)

transfer_model = transfer_model.to(device)

print(transfer_model.classifier)


In [ ]:
TRANSFER_EPOCHS = 5
TRANSFER_LR = 0.001

transfer_model, transfer_history, transfer_training_time = train_model(
    transfer_model,
    train_loader,
    val_loader,
    epochs=TRANSFER_EPOCHS,
    learning_rate=TRANSFER_LR
)

print(f"MobileNetV2 training time: {transfer_training_time:.2f} seconds")


In [ ]:
plot_history(transfer_history, "MobileNetV2 Transfer Learning")


## 10. Evaluation Function

Accuracy alone is not enough for a multi-class classifier, so precision, recall and F1-score are also calculated.

The same test set is used for both models.

In [ ]:
def evaluate_model(model, loader):
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(
        all_labels, all_predictions,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        all_labels, all_predictions,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        all_labels, all_predictions,
        average="weighted",
        zero_division=0
    )

    return accuracy, precision, recall, f1, all_labels, all_predictions


custom_accuracy, custom_precision, custom_recall, custom_f1, custom_y_true, custom_y_pred = evaluate_model(
    custom_model, test_loader
)

transfer_accuracy, transfer_precision, transfer_recall, transfer_f1, transfer_y_true, transfer_y_pred = evaluate_model(
    transfer_model, test_loader
)

print("Custom CNN")
print(classification_report(custom_y_true, custom_y_pred, target_names=class_names, zero_division=0))

print("MobileNetV2")
print(classification_report(transfer_y_true, transfer_y_pred, target_names=class_names, zero_division=0))


## 11. Model Size

Model size is measured from the saved PyTorch state dictionary. This is more useful for deployment than simply comparing the number of layers because it represents the approximate weight-file size that needs to be stored.

In [ ]:
def get_model_size_mb(model, file_name):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    return size_mb


custom_size = get_model_size_mb(custom_model, "custom_cnn_weights.pth")
transfer_size = get_model_size_mb(transfer_model, "mobilenetv2_weights.pth")

print(f"Custom CNN size: {custom_size:.2f} MB")
print(f"MobileNetV2 size: {transfer_size:.2f} MB")


## 12. Inference Speed

Inference speed is measured using batch size 1 because the deployment application will normally receive one uploaded image at a time.

A few warm-up predictions are performed first so the timing is more stable.

In [ ]:
def measure_inference_speed(model, loader, num_images=100):
    model.eval()

    images_seen = 0

    # Warm-up
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            _ = model(images)

            images_seen += images.size(0)
            if images_seen >= 10:
                break

    single_images = []

    for images, _ in loader:
        for image in images:
            single_images.append(image)
            if len(single_images) >= num_images:
                break

        if len(single_images) >= num_images:
            break

    start_time = time.perf_counter()

    with torch.no_grad():
        for image in single_images:
            image = image.unsqueeze(0).to(device)
            _ = model(image)

    elapsed = time.perf_counter() - start_time

    average_ms = (elapsed / len(single_images)) * 1000
    images_per_second = len(single_images) / elapsed

    return average_ms, images_per_second


# A separate loader with batch size 1 makes the deployment-style test clear.
speed_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

custom_ms, custom_ips = measure_inference_speed(custom_model, speed_loader)
transfer_ms, transfer_ips = measure_inference_speed(transfer_model, speed_loader)

print(f"Custom CNN: {custom_ms:.2f} ms/image | {custom_ips:.2f} images/sec")
print(f"MobileNetV2: {transfer_ms:.2f} ms/image | {transfer_ips:.2f} images/sec")


## 13. Final Comparison

The table below brings the four requested measurements together.

The training time is hardware-dependent, so it should be interpreted as the time measured on the machine used to run this notebook. The same hardware should be used for both models for a fair comparison.

In [ ]:
results = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2"],
    "Accuracy": [custom_accuracy, transfer_accuracy],
    "Model Size (MB)": [custom_size, transfer_size],
    "Training Time (seconds)": [custom_training_time, transfer_training_time],
    "Inference Time (ms/image)": [custom_ms, transfer_ms],
    "Inference Speed (images/sec)": [custom_ips, transfer_ips]
})

display(results)


## 14. Deployment Recommendation

For a deployment model, accuracy is important, but it is not the only consideration. A smaller model can reduce storage requirements and a faster model can make interactive prediction feel more responsive.

The recommendation below uses the measured results rather than assuming that one architecture is always better.

A model is considered attractive for deployment when:

1. Its accuracy is close to the other model.
2. Its model file is smaller.
3. Its single-image inference is fast.
4. It does not give up a large amount of classification performance just to reduce size.

In [ ]:
accuracy_difference = transfer_accuracy - custom_accuracy
size_difference = transfer_size - custom_size
speed_difference = transfer_ms - custom_ms

# Simple deployment rule:
# If MobileNetV2 is within 1 percentage point of the custom CNN
# and is smaller, prefer it for deployment.
if accuracy_difference >= -0.01 and transfer_size < custom_size:
    recommendation = "MobileNetV2"
    reason = (
        "MobileNetV2 gives similar test accuracy while using a smaller "
        "weight file. This makes it a practical choice for an interactive "
        "deployment, especially when storage and response time matter."
    )
else:
    recommendation = "Custom CNN"
    reason = (
        "The custom CNN provides a useful accuracy advantage without a "
        "large enough deployment benefit from MobileNetV2 in this experiment."
    )

print("Recommended deployment model:", recommendation)
print()
print(reason)
print()
print("Accuracy difference (MobileNetV2 - Custom CNN):", f"{accuracy_difference * 100:.2f} percentage points")
print("Size difference (MobileNetV2 - Custom CNN):", f"{size_difference:.2f} MB")
print("Inference time difference (MobileNetV2 - Custom CNN):", f"{speed_difference:.2f} ms/image")


### Analytical conclusion

The final recommendation should be based on the actual table above.

**Accuracy:** Compare whether transfer learning improves or reduces test accuracy.

**Model size:** A smaller weight file is easier to store and distribute.

**Training time:** Transfer learning normally needs less training than building a deep visual representation completely from scratch because the backbone already contains ImageNet features. However, the exact time depends on GPU/CPU hardware.

**Inference speed:** Measure this on the same device and with the same batch size. For a student-facing application that predicts one image at a time, single-image latency is particularly useful.

If MobileNetV2 maintains almost the same accuracy while being smaller and sufficiently fast, it is a strong deployment choice. If the custom CNN is clearly more accurate and the size/speed difference is small, the custom CNN can be justified instead.

## 15. Save the Models

These files are useful if the trained models will be used later in a small application.

The notebook saves only the weights, not the complete Python environment.

In [ ]:
torch.save(custom_model.state_dict(), "custom_cnn_final.pth")
torch.save(transfer_model.state_dict(), "mobilenetv2_final.pth")

print("Saved:")
print("- custom_cnn_final.pth")
print("- mobilenetv2_final.pth")


## 16. Final Summary

This bonus project demonstrates:

- A custom CNN baseline
- MobileNetV2 transfer learning with ImageNet weights
- The same dataset split for both models
- Accuracy comparison
- Model-size comparison
- Training-time comparison
- Single-image inference-speed comparison
- A deployment recommendation based on measured results

### Important limitation

Training and inference timings depend on the hardware used to run the notebook. For the final submission, run the complete notebook on the same hardware and keep the generated comparison table and recommendation from that run.

For the cleanest final result, Google Colab with a GPU is recommended.